## 测试补完输出服务

In [1]:
import requests
import base64
import os
from typing import Dict, List, Optional

# ====================== 配置项（按需修改）======================
IP = "10.120.17.131"  # 服务器IP
PORT = 8000            # 服务端口
MODEL_NAME = "Qwen/Qwen3-VL-2B-Instruct"
TIMEOUT = 60           # 超时时间（秒）
ALLOWED_IMAGE_FORMATS = ["jpg", "jpeg", "png", "bmp"]  # 支持的图片格式

# 服务器地址
SERVER_URL = f"http://{IP}:{PORT}/v1/chat/completions"
HEALTH_CHECK_URL = f"http://{IP}:{PORT}/health"

def image_to_base64(image_path: str) -> str:
    """
    将图片转为base64编码（增加格式校验和异常处理）
    """
    # 1. 校验文件是否存在
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"图片文件不存在：{image_path}")
    
    # 2. 校验图片格式
    file_ext = image_path.split(".")[-1].lower()
    if file_ext not in ALLOWED_IMAGE_FORMATS:
        raise ValueError(
            f"不支持的图片格式：{file_ext}，仅支持 {ALLOWED_IMAGE_FORMATS}"
        )
    
    # 3. 读取并编码
    try:
        with open(image_path, "rb") as f:
            return base64.b64encode(f.read()).decode("utf-8")
    except Exception as e:
        raise RuntimeError(f"图片编码失败：{str(e)}")

def check_server_health() -> bool:
    """
    检查服务器健康状态（调用前先验证服务是否可用）
    """
    try:
        response = requests.get(HEALTH_CHECK_URL, timeout=10)
        if response.status_code == 200:
            health_info = response.json()
            print(f"✅ 服务器健康状态：{health_info}")
            return health_info.get("status") == "healthy" and health_info.get("model_loaded")
        else:
            print(f"❌ 服务器健康检查失败，状态码：{response.status_code}")
            return False
    except Exception as e:
        print(f"❌ 无法连接到服务器：{str(e)}")
        return False

def call_qwen_vl(
    text_prompt: str,
    image_path: Optional[str] = None  # 可选：不传图片则为纯文本调用
) -> str:
    """
    调用Qwen3-VL服务（支持纯文本/图文混合模式）
    """
    # 1. 先检查服务器健康状态
    if not check_server_health():
        return "❌ 服务器未就绪，无法调用"
    
    # 2. 构造请求内容
    content = [{"type": "text", "text": text_prompt}]
    
    # 3. 如有图片则添加base64编码的图片
    if image_path:
        try:
            image_b64 = image_to_base64(image_path)
            content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/{image_path.split('.')[-1].lower()};base64,{image_b64}"}
            })
            print(f"📸 已加载图片：{image_path}")
        except Exception as e:
            return f"❌ 图片处理失败：{str(e)}"
    
    # 4. 构造完整请求体
    payload: Dict = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": content}],
        "max_tokens": 1024,
        "temperature": 0.7,
        "top_p": 0.8,  # 新增：适配Qwen3-VL的采样参数
        "repetition_penalty": 1.05  # 新增：减少重复生成
    }
    
    # 5. 发送请求并处理响应
    try:
        print(f"📡 正在调用服务器：{SERVER_URL}")
        response = requests.post(
            SERVER_URL,
            json=payload,
            headers={"Content-Type": "application/json"},
            timeout=TIMEOUT
        )
        
        # 打印完整响应（便于调试）
        print(f"🔍 响应状态码：{response.status_code}")
        print(f"🔍 原始响应内容：{response.text}")
        
        response.raise_for_status()  # 抛出HTTP错误
        result = response.json()
        
        # 6. 解析回复
        return result["choices"][0]["message"]["content"]
    
    except requests.exceptions.HTTPError as e:
        return f"❌ HTTP请求失败：{str(e)}\n响应详情：{response.text if 'response' in locals() else '无'}"
    except requests.exceptions.Timeout:
        return f"❌ 请求超时（超过{TIMEOUT}秒）"
    except requests.exceptions.ConnectionError:
        return f"❌ 无法连接到服务器 {IP}:{PORT}，请检查网络或端口是否开放"
    except KeyError as e:
        return f"❌ 响应解析失败（字段缺失）：{str(e)}\n原始响应：{response.text if 'response' in locals() else '无'}"
    except Exception as e:
        return f"❌ 调用异常：{str(e)}"

if __name__ == "__main__":
    # ====================== 测试模式选择（按需注释/取消注释）======================
    # 模式1：图文混合调用（默认）
    test_image_path = "test.png"  # 替换为你的图片路径
    test_prompt = "请详细描述这张图片的内容，包括物体、颜色、场景等"
    
    # 模式2：纯文本调用（调试用，排除图片问题）
    # test_image_path = None
    # test_prompt = "你好，请介绍一下自己的功能和特点"
    
    # ====================== 执行调用并打印结果 ======================
    print("=" * 50)
    print("📝 提问内容：")
    print(test_prompt)
    if test_image_path:
        print(f"🖼️ 图片路径：{test_image_path}")
    print("=" * 50)
    
    # 调用服务
    reply = call_qwen_vl(test_prompt, test_image_path)
    
    print("\n🤖 Qwen3-VL 回复：")
    print(reply)
    print("=" * 50)

📝 提问内容：
请详细描述这张图片的内容，包括物体、颜色、场景等
🖼️ 图片路径：test.png
✅ 服务器健康状态：{'status': 'healthy', 'model_loaded': True, 'gpu_count': 4, 'cuda_available': True, 'model_name': 'Qwen/Qwen3-VL-2B-Instruct'}
📸 已加载图片：test.png
📡 正在调用服务器：http://10.120.17.131:8000/v1/chat/completions
🔍 响应状态码：200
🔍 原始响应内容：{"id":"chat-820998","object":"chat.completion","created":1769364655,"model":"Qwen/Qwen3-VL-2B-Instruct","choices":[{"index":0,"message":{"role":"assistant","content":"这是一张充满活力和喜悦感的户外照片。画面中央是一只毛茸茸的小白狗，它正开心地在草地上奔跑。小狗的毛发洁白蓬松，像一团云朵，显得格外可爱。它的耳朵竖立着，眼睛圆溜溜的，透着明亮的光泽，嘴巴微微张开，露出粉红色的舌头，仿佛在欢快地叫唤或玩耍。它脖子上系着一条蓝色的带子，可能是宠物项圈或者小围巾。小狗的一只前爪抬起，似乎是在向镜头打招呼或庆祝。\n\n背景是模糊的森林或树林，可以看到绿色和棕色的色调，营造出一种自然、宁静的氛围。前景中，翠绿的草地清晰可见，与小狗的白色皮毛形成鲜明对比。阳光从上方照射下来，在草叶和小狗身上投下柔和的光影，使整个画面显得温暖而生动。整张图片充满了动感和快乐的气息，捕捉到了一个天真烂漫的瞬间。\n"},"finish_reason":"stop"}],"usage":{"prompt_tokens":438,"completion_tokens":212,"total_tokens":650}}

🤖 Qwen3-VL 回复：
这是一张充满活力和喜悦感的户外照片。画面中央是一只毛茸茸的小白狗，它正开心地在草地上奔跑。小狗的毛发洁白蓬松，像一团云朵，显得格外可爱。它的耳朵竖立着，眼睛圆溜溜的，透着明亮的光泽，嘴巴微微张开，露出粉红色的舌头，仿佛在欢快地叫唤或玩耍。它脖

## 测试流式输出服务

In [5]:
import requests
import base64
import os
import json
from typing import Dict, List, Optional

# ====================== 配置项（按需修改）======================
IP = "10.120.17.131"  # 服务器IP
PORT = 8000            # 服务端口
MODEL_NAME = "Qwen/Qwen3-VL-2B-Instruct"
TIMEOUT = 60           # 超时时间（秒）
ALLOWED_IMAGE_FORMATS = ["jpg", "jpeg", "png", "bmp"]  # 支持的图片格式

# 服务器地址
SERVER_URL = f"http://{IP}:{PORT}/v1/chat/completions/stream"
HEALTH_CHECK_URL = f"http://{IP}:{PORT}/health"

def image_to_base64(image_path: str) -> str:
    """
    将图片转为base64编码（增加格式校验和异常处理）
    """
    # 1. 校验文件是否存在
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"图片文件不存在：{image_path}")
    
    # 2. 校验图片格式
    file_ext = image_path.split(".")[-1].lower()
    if file_ext not in ALLOWED_IMAGE_FORMATS:
        raise ValueError(
            f"不支持的图片格式：{file_ext}，仅支持 {ALLOWED_IMAGE_FORMATS}"
        )
    
    # 3. 读取并编码
    try:
        with open(image_path, "rb") as f:
            return base64.b64encode(f.read()).decode("utf-8")
    except Exception as e:
        raise RuntimeError(f"图片编码失败：{str(e)}")

def check_server_health() -> bool:
    """
    检查服务器健康状态（调用前先验证服务是否可用）
    """
    try:
        response = requests.get(HEALTH_CHECK_URL, timeout=10)
        if response.status_code == 200:
            health_info = response.json()
            print(f"✅ 服务器健康状态：{health_info}")
            return health_info.get("status") == "healthy" and health_info.get("model_loaded")
        else:
            print(f"❌ 服务器健康检查失败，状态码：{response.status_code}")
            return False
    except Exception as e:
        print(f"❌ 无法连接到服务器：{str(e)}")
        return False

def call_qwen_vl(
    text_prompt: str,
    image_path: Optional[str] = None  # 可选：不传图片则为纯文本调用
) -> str:
    """
    调用Qwen3-VL服务（适配流式输出，保持原有接口返回字符串）
    """
    # 1. 先检查服务器健康状态
    if not check_server_health():
        return "❌ 服务器未就绪，无法调用"
    
    # 2. 构造请求内容
    content = [{"type": "text", "text": text_prompt}]
    
    # 3. 如有图片则添加base64编码的图片
    if image_path:
        try:
            image_b64 = image_to_base64(image_path)
            content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/{image_path.split('.')[-1].lower()};base64,{image_b64}"}
            })
            print(f"📸 已加载图片：{image_path}")
        except Exception as e:
            return f"❌ 图片处理失败：{str(e)}"
    
    # 4. 构造完整请求体（新增stream=True参数）
    payload: Dict = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": content}],
        "max_tokens": 1024,
        "temperature": 0.7,
        "top_p": 0.8,
        "repetition_penalty": 1.05,
        "stream": True  # 开启流式输出
    }
    
    # 5. 发送请求并处理流式响应
    try:
        print(f"📡 正在调用服务器：{SERVER_URL}")
        response = requests.post(
            SERVER_URL,
            json=payload,
            headers={"Content-Type": "application/json"},
            timeout=TIMEOUT,
            stream=True  # 关键：启用流式响应
        )
        
        # 打印响应状态（保持原有调试信息）
        print(f"🔍 响应状态码：{response.status_code}")
        
        response.raise_for_status()  # 抛出HTTP错误
        
        # 6. 处理流式响应
        full_response = ""  # 拼接完整回复
        raw_response_lines = []  # 保存原始响应行用于调试
        
        # 逐行读取流式响应
        for line in response.iter_lines():
            if line:
                # 解码并清理行内容
                line_str = line.decode("utf-8").strip()
                raw_response_lines.append(line_str)
                
                # 跳过空行和结束标记
                if not line_str or line_str == "data: [DONE]":
                    continue
                
                # 解析SSE格式数据
                if line_str.startswith("data: "):
                    json_str = line_str[6:]  # 去掉"data: "前缀
                    try:
                        # 解析JSON块
                        chunk = json.loads(json_str)
                        
                        # 处理错误响应
                        if "error" in chunk:
                            return f"❌ 流式调用出错：{chunk['error']['message']}"
                        
                        # 提取生成的内容
                        if (chunk.get("choices") and 
                            len(chunk["choices"]) > 0 and 
                            "delta" in chunk["choices"][0] and 
                            "content" in chunk["choices"][0]["delta"]):
                            
                            content = chunk["choices"][0]["delta"]["content"]
                            full_response += content
                            # 实时打印（可选，注释掉则只返回最终结果）
                            print(content, end="", flush=True)
                            
                    except json.JSONDecodeError:
                        continue
        
        # 打印原始响应（保持原有调试逻辑）
        print(f"\n🔍 原始响应内容：\n{chr(10).join(raw_response_lines)}")
        
        # 7. 返回完整拼接的结果（保持原有返回格式）
        if full_response:
            return full_response
        else:
            return "❌ 未获取到有效回复"
    
    except requests.exceptions.HTTPError as e:
        error_detail = response.text if 'response' in locals() else '无'
        return f"❌ HTTP请求失败：{str(e)}\n响应详情：{error_detail}"
    except requests.exceptions.Timeout:
        return f"❌ 请求超时（超过{TIMEOUT}秒）"
    except requests.exceptions.ConnectionError:
        return f"❌ 无法连接到服务器 {IP}:{PORT}，请检查网络或端口是否开放"
    except Exception as e:
        return f"❌ 调用异常：{str(e)}"

if __name__ == "__main__":
    # ====================== 测试模式选择（按需注释/取消注释）======================
    # 模式1：图文混合调用（默认）
    test_image_path = "test.png"  # 替换为你的图片路径
    test_prompt = "请详细描述这张图片的内容，包括物体、颜色、场景等"
    
    # 模式2：纯文本调用（调试用，排除图片问题）
    # test_image_path = None
    # test_prompt = "你好，请介绍一下自己的功能和特点"
    
    # ====================== 执行调用并打印结果 ======================
    print("=" * 50)
    print("📝 提问内容：")
    print(test_prompt)
    if test_image_path:
        print(f"🖼️ 图片路径：{test_image_path}")
    print("=" * 50)
    
    # 调用服务（流式输出）
    print("\n🤖 Qwen3-VL 回复：")
    reply = call_qwen_vl(test_prompt, test_image_path)
    
    # 如果实时打印了内容，这里只做最终确认；否则打印完整回复
    if not reply.startswith("❌") and "\n" not in reply:
        print("\n" + "=" * 50)
    else:
        print(reply)
        print("=" * 50)

📝 提问内容：
请详细描述这张图片的内容，包括物体、颜色、场景等
🖼️ 图片路径：test.png

🤖 Qwen3-VL 回复：
✅ 服务器健康状态：{'status': 'healthy', 'model_loaded': True, 'gpu_count': 4, 'cuda_available': True, 'model_name': 'Qwen/Qwen3-VL-2B-Instruct'}
📸 已加载图片：test.png
📡 正在调用服务器：http://10.120.17.131:8000/v1/chat/completions/stream
🔍 响应状态码：200
一张充满活力和喜悦感的户外照片面中心是一只毛茸茸的白色小狗，它正站在一片绿色的草地上，似乎正在欢快地奔跑或跳跃。

- 体只小狗体型娇小，拥有浓密蓬松的白色长毛，显得非常可爱。它的耳朵竖立着，眼睛大而明亮，呈深邃的黑色，充满了好奇与兴奋。它的嘴巴微微张开，粉红色的舌头伸出，露出一个灿烂的笑容，仿佛在开心地叫唤或玩耍。它的前爪抬起，呈现出一种活泼的姿态。
- **配饰**：小狗脖子上系着一条蓝色的带子，上面有白色的图案，像是一个小巧的装饰品，为它增添了俏皮感。
- **背景与环境**：背景是模糊的树林，可以看到一些树木的轮廓和柔和的光斑，营造出一种温暖、自然的感觉。阳光透过树叶洒下，在草地上形成斑驳的光影。
- **色彩与氛围**：整张照片以白色为主色调，点缀着草地的翠绿和天空的淡蓝，以及小狗身上的蓝色领巾。整体色调明亮清新，光线充足，给人一种生机勃勃、快乐无忧的感觉
🔍 原始响应内容：
data: {"id": "chat-648095", "object": "chat.completion.chunk", "created": 1769364795, "model": "Qwen/Qwen3-VL-2B-Instruct", "choices": [{"index": 0, "delta": {"content": "一"}, "finish_reason": null}]}
data: {"id": "chat-648095", "object": "chat.completion.chunk", "created": 1769364795, "model": "Qwen/Qwen

## 测试补全和流式输出的多图片情况

In [3]:
import requests
import base64
import os
import json
from typing import Dict, List, Optional

# ====================== 配置项（按需修改）======================
IP = "10.120.17.131"  # 服务器IP
PORT = 8000            # 服务端口
MODEL_NAME = "Qwen/Qwen3-VL-2B-Instruct"
TIMEOUT = 60           # 超时时间（秒）
ALLOWED_IMAGE_FORMATS = ["jpg", "jpeg", "png", "bmp"]  # 支持的图片格式

# 服务器地址
SERVER_URL_STREAM = f"http://{IP}:{PORT}/v1/chat/completions/stream"
SERVER_URL_COMPLETION = f"http://{IP}:{PORT}/v1/chat/completions"
HEALTH_CHECK_URL = f"http://{IP}:{PORT}/health"

def image_to_base64(image_path: str) -> str:
    """将单张图片转为base64编码"""
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"图片文件不存在：{image_path}")
    file_ext = image_path.split(".")[-1].lower()
    if file_ext not in ALLOWED_IMAGE_FORMATS:
        raise ValueError(f"不支持的图片格式：{file_ext}")
    try:
        with open(image_path, "rb") as f:
            return base64.b64encode(f.read()).decode("utf-8")
    except Exception as e:
        raise RuntimeError(f"图片编码失败：{str(e)}")

def check_server_health() -> bool:
    """检查服务器健康状态"""
    try:
        response = requests.get(HEALTH_CHECK_URL, timeout=10)
        if response.status_code == 200:
            health_info = response.json()
            print(f"✅ 服务器健康状态：{health_info}")
            return health_info.get("status") == "healthy" and health_info.get("model_loaded")
        else:
            print(f"❌ 服务器健康检查失败，状态码：{response.status_code}")
            return False
    except Exception as e:
        print(f"❌ 无法连接到服务器：{str(e)}")
        return False

def call_qwen_vl_multi_image(
    text_prompt: str,
    image_paths: Optional[List[str]] = None,  # 支持多张图片
    stream: bool = True  # 控制流式/非流式
) -> str:
    """
    调用Qwen3-VL服务（支持多图输入 + 流式/非流式切换）
    """
    if not check_server_health():
        return "❌ 服务器未就绪，无法调用"
    
    # 1. 构造基础content（文本 + 多张图片）
    content = [{"type": "text", "text": text_prompt}]
    if image_paths and len(image_paths) > 0:
        for idx, img_path in enumerate(image_paths):
            try:
                img_b64 = image_to_base64(img_path)
                img_ext = img_path.split(".")[-1].lower()
                content.append({
                    "type": "image_url",
                    "image_url": {"url": f"data:image/{img_ext};base64,{img_b64}"}
                })
                print(f"📸 已加载图片 {idx+1}/{len(image_paths)}：{img_path}")
            except Exception as e:
                return f"❌ 图片 {img_path} 处理失败：{str(e)}"
    
    # 2. 构造请求体
    payload: Dict = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": content}],
        "max_tokens": 2048,  # 多图场景建议增大token上限
        "temperature": 0.7,
        "top_p": 0.8,
        "repetition_penalty": 1.05,
        "stream": stream  # 切换流式/非流式
    }
    
    # 3. 选择请求地址
    server_url = SERVER_URL_STREAM if stream else SERVER_URL_COMPLETION
    print(f"📡 正在调用服务器：{server_url}")

    try:
        response = requests.post(
            server_url,
            json=payload,
            headers={"Content-Type": "application/json"},
            timeout=TIMEOUT,
            stream=stream  # 流式需开启stream参数
        )
        response.raise_for_status()
        print(f"🔍 响应状态码：{response.status_code}")

        # 4. 处理流式响应
        if stream:
            full_response = ""
            for line in response.iter_lines():
                if line:
                    line_str = line.decode("utf-8").strip()
                    if line_str.startswith("data: ") and line_str != "data: [DONE]":
                        json_str = line_str[6:]
                        chunk = json.loads(json_str)
                        if "choices" in chunk and chunk["choices"][0]["delta"].get("content"):
                            content_chunk = chunk["choices"][0]["delta"]["content"]
                            full_response += content_chunk
                            print(content_chunk, end="", flush=True)
            print("\n")
            return full_response
        # 5. 处理非流式响应
        else:
            result = response.json()
            return result["choices"][0]["message"]["content"]

    except Exception as e:
        return f"❌ 调用异常：{str(e)}"

if __name__ == "__main__":
    # ====================== 测试配置 =======================
    # 多图输入示例
    test_image_paths = ["test.png", "test1.jpg"]  # 替换为你的多张图片路径
    test_prompt = "请详细描述这两张图片的内容，并对比它们的异同点"
    
    # 模式1：流式输出（默认）
    print("======= 流式多图调用 =======")
    reply_stream = call_qwen_vl_multi_image(test_prompt, test_image_paths, stream=True)
    
    # 模式2：非流式输出
    # print("======= 非流式多图调用 =======")
    # reply_completion = call_qwen_vl_multi_image(test_prompt, test_image_paths, stream=False)
    # print("🤖 非流式回复：\n", reply_completion)

======= 流式多图调用 =======
✅ 服务器健康状态：{'status': 'healthy', 'model_loaded': True, 'gpu_count': 4, 'cuda_available': True, 'model_name': 'Qwen/Qwen3-VL-2B-Instruct'}
📸 已加载图片 1/2：test.png
📸 已加载图片 2/2：test1.jpg
📡 正在调用服务器：http://10.120.17.131:8000/v1/chat/completions/stream
🔍 响应状态码：200
，这是一份关于您的两张图片的详细描述及其异同点分析。

### 图片内容描述**第一张图片：**
这张照片捕捉了一只白色小狗在户外阳光下的生动瞬间。这只小狗体型娇小，毛茸茸的白色皮毛蓬松而柔软，在阳光下显得格外耀眼。它的眼睛圆溜溜、黑亮，充满着好奇与喜悦，正对着镜头开心地笑着，嘴巴微微张开，露出粉色的舌头和洁白的牙齿。它的前爪抬起，仿佛正在欢快地奔跑或跳跃，充满了活力与动感。小狗脖子上系着一条蓝色的带子，背景是模糊的绿色树林，营造出一种清新自然的氛围。整个画面构图简洁，焦点清晰，突出了小狗活泼可爱的神态。

**第二张图片：**
这张照片展示了一只戴着草帽的灰色猫咪，它正站在一片开满粉红色花朵的草坪上。猫咪有着灰白色的虎斑纹毛发和一双明亮的绿眼睛，表情略显呆萌，直视着镜头。它身上佩戴着一个黄色的牵引背带，连接着一根黄色的牵引绳，表明它可能是在被牵着散步或玩耍。背景是茂密的粉红色花丛，为画面增添了生机与色彩。整体色调柔和，光线充足，营造出一种宁静而美好的夏日氛围。

---

### 异同点对比| 对比维度| 第一张图片（小狗） | 第二张图片（猫咪） |
| :--- | :--- | |
| **主体动物** | 一只白色的小狗| 一只灰色的猫 |
| **动物特征** | 身材小巧，毛色纯白，眼神灵动，姿态活泼| 毛色灰白，有明显的虎斑纹，眼神警觉，神情温和 |
| **服饰/配饰** | 穿着一条蓝色的带子（可能是项圈或围巾），但未见其他装饰| 头戴一顶编织草帽，身穿黄色的牵引背带 |
| **环境** | 青翠的草地和模糊的树林，阳光明媚| 开满粉红花朵的草坪，背景有繁茂的植物 |
| **拍摄风格** | 动感

In [4]:
import requests
import base64
import os
import json
from typing import Dict, List, Optional

# ====================== 配置项（按需修改）======================
IP = "10.120.17.131"  # 服务器IP
PORT = 8000            # 服务端口
MODEL_NAME = "Qwen/Qwen3-VL-2B-Instruct"
TIMEOUT = 60           # 超时时间（秒）
ALLOWED_IMAGE_FORMATS = ["jpg", "jpeg", "png", "bmp"]  # 支持的图片格式

# 服务器地址
SERVER_URL_STREAM = f"http://{IP}:{PORT}/v1/chat/completions/stream"
SERVER_URL_COMPLETION = f"http://{IP}:{PORT}/v1/chat/completions"
HEALTH_CHECK_URL = f"http://{IP}:{PORT}/health"

def image_to_base64(image_path: str) -> str:
    """将单张图片转为base64编码"""
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"图片文件不存在：{image_path}")
    file_ext = image_path.split(".")[-1].lower()
    if file_ext not in ALLOWED_IMAGE_FORMATS:
        raise ValueError(f"不支持的图片格式：{file_ext}")
    try:
        with open(image_path, "rb") as f:
            return base64.b64encode(f.read()).decode("utf-8")
    except Exception as e:
        raise RuntimeError(f"图片编码失败：{str(e)}")

def check_server_health() -> bool:
    """检查服务器健康状态"""
    try:
        response = requests.get(HEALTH_CHECK_URL, timeout=10)
        if response.status_code == 200:
            health_info = response.json()
            print(f"✅ 服务器健康状态：{health_info}")
            return health_info.get("status") == "healthy" and health_info.get("model_loaded")
        else:
            print(f"❌ 服务器健康检查失败，状态码：{response.status_code}")
            return False
    except Exception as e:
        print(f"❌ 无法连接到服务器：{str(e)}")
        return False

def call_qwen_vl_multi_image(
    text_prompt: str,
    image_paths: Optional[List[str]] = None,  # 支持多张图片
    stream: bool = True  # 控制流式/非流式
) -> str:
    """
    调用Qwen3-VL服务（支持多图输入 + 流式/非流式切换）
    """
    if not check_server_health():
        return "❌ 服务器未就绪，无法调用"
    
    # 1. 构造基础content（文本 + 多张图片）
    content = [{"type": "text", "text": text_prompt}]
    if image_paths and len(image_paths) > 0:
        for idx, img_path in enumerate(image_paths):
            try:
                img_b64 = image_to_base64(img_path)
                img_ext = img_path.split(".")[-1].lower()
                content.append({
                    "type": "image_url",
                    "image_url": {"url": f"data:image/{img_ext};base64,{img_b64}"}
                })
                print(f"📸 已加载图片 {idx+1}/{len(image_paths)}：{img_path}")
            except Exception as e:
                return f"❌ 图片 {img_path} 处理失败：{str(e)}"
    
    # 2. 构造请求体
    payload: Dict = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": content}],
        "max_tokens": 2048,  # 多图场景建议增大token上限
        "temperature": 0.7,
        "top_p": 0.8,
        "repetition_penalty": 1.05,
        "stream": stream  # 切换流式/非流式
    }
    
    # 3. 选择请求地址
    server_url = SERVER_URL_STREAM if stream else SERVER_URL_COMPLETION
    print(f"📡 正在调用服务器：{server_url}")

    try:
        response = requests.post(
            server_url,
            json=payload,
            headers={"Content-Type": "application/json"},
            timeout=TIMEOUT,
            stream=stream  # 流式需开启stream参数
        )
        response.raise_for_status()
        print(f"🔍 响应状态码：{response.status_code}")

        # 4. 处理流式响应
        if stream:
            full_response = ""
            for line in response.iter_lines():
                if line:
                    line_str = line.decode("utf-8").strip()
                    if line_str.startswith("data: ") and line_str != "data: [DONE]":
                        json_str = line_str[6:]
                        chunk = json.loads(json_str)
                        if "choices" in chunk and chunk["choices"][0]["delta"].get("content"):
                            content_chunk = chunk["choices"][0]["delta"]["content"]
                            full_response += content_chunk
                            print(content_chunk, end="", flush=True)
            print("\n")
            return full_response
        # 5. 处理非流式响应
        else:
            result = response.json()
            return result["choices"][0]["message"]["content"]

    except Exception as e:
        return f"❌ 调用异常：{str(e)}"

if __name__ == "__main__":
    # ====================== 测试配置 =======================
    # 多图输入示例
    test_image_paths = ["test.png", "test1.jpg"]  # 替换为你的多张图片路径
    test_prompt = "请详细描述这两张图片的内容，并对比它们的异同点"
    
    # # 模式1：流式输出（默认）
    # print("======= 流式多图调用 =======")
    # reply_stream = call_qwen_vl_multi_image(test_prompt, test_image_paths, stream=True)
    
    # 模式2：非流式输出
    print("======= 非流式多图调用 =======")
    reply_completion = call_qwen_vl_multi_image(test_prompt, test_image_paths, stream=False)
    print("🤖 非流式回复：\n", reply_completion)

======= 非流式多图调用 =======
✅ 服务器健康状态：{'status': 'healthy', 'model_loaded': True, 'gpu_count': 4, 'cuda_available': True, 'model_name': 'Qwen/Qwen3-VL-2B-Instruct'}
📸 已加载图片 1/2：test.png
📸 已加载图片 2/2：test1.jpg
📡 正在调用服务器：http://10.120.17.131:8000/v1/chat/completions


🔍 响应状态码：200
🤖 非流式回复：
 好的，这是对两张图片的详细描述及其异同点分析。

---

### 图片一：白色小狗在草地上奔跑

这张照片捕捉了一只毛茸茸的白色小狗在户外草地上的生动瞬间。画面主体是一只体型娇小、毛发蓬松的白色狗狗，它正朝着镜头方向跑来，前爪抬起，显得充满活力和喜悦。它的耳朵竖立，眼睛明亮有神，嘴巴微张，露出粉红色的舌头，似乎正在开心地叫唤或玩耍。狗狗脖子上系着一条蓝色的布带，为纯白的毛色增添了一抹亮色。背景是模糊的绿色植物和树木轮廓，采用了浅景深效果，突出了前景中的小狗。整体光线柔和自然，营造出一种清新、活泼的氛围。

### 图片二：戴着帽子的猫咪

这张照片展示了一只灰色的英国短毛猫（或称苏格兰折耳猫），它正站在一片开满粉色花朵的草地上。这只猫最引人注目的特征是它头上戴着一顶编织的草帽，增添了几分田园风情。它的眼睛大而圆，瞳孔呈绿色，直视着镜头，表情略显严肃或好奇。猫的身体被一个黄色的牵引背带固定住，这表明它可能是在公园或花园里散步时被牵着。背景中繁茂的粉色花朵与翠绿的草地相映成趣，构成了一幅宁静而美丽的画面。整个场景充满了夏日的气息。

---

### 异同点对比

| 特征 | 图片一 (小狗) | 图片二 (猫咪) |
| :--- | :--- | :--- |
| **主题** | 动物（狗） | 动物（猫） |
| **物种** | 一只小型犬（可能是博美或类似品种） | 一只长毛猫（可能是英国短毛猫） |
| **姿态** | 奔跑、跳跃，充满动感 | 站立，面向镜头，姿态静态 |
| **服饰/配饰** | 蓝色项圈 | 编织草帽 + 黄色牵引带 |
| **环境** | 森林/树林边缘，地面覆盖着青草 | 开花的草坪，周围有花卉 |
| **色彩基调** | 主要以白色为主，点缀蓝色 | 主要由灰、绿、粉构成，视觉焦点集中于猫的面部 |
| **情感表达** | 充满快乐、兴奋和活力 | 表情平静、专注，带有好奇心 |
| **构图** | 采用特写+浅景深，突出主体 | 采用中近景，完整呈现动物与环境的关系 |

总的来说，两幅图像都展现了宠物动物在自然环境中愉快的状态，但它们的题材、细节和表现方式截然不同。第一张图强调的是动态和生命力，而第二张图则更注重于动物的可爱造型和与环境的和谐共处。



## 定义为类来使用

In [ ]:
from client import PhysicalAICenterClient

if __name__ == "__main__":
    # ====================== 测试配置 =======================
    # 初始化客户端（可自定义配置参数）
    client = PhysicalAICenterClient(
        ip="10.120.17.131",
        port=8000,
        model_name="Qwen/Qwen3-VL-30B-A3B-Instruct",
        timeout=60,
        max_tokens=2048
    )
    
    # 多图输入示例
    test_image_paths = ["test.png", "test1.jpg"]  # 替换为你的多张图片路径
    test_prompt = "请详细描述这两张图片的内容，并对比它们的异同点"
    
    # 模式1：流式输出（默认）
    print("======= 流式多图调用 =======")
    reply_stream = client.call_qwen_vl_multi_image(test_prompt, test_image_paths, stream=True)
    
    # 模式2：非流式输出
    # print("======= 非流式多图调用 =======")
    # reply_completion = client.call_qwen_vl_multi_image(test_prompt, test_image_paths, stream=False)
    # print("🤖 非流式回复：\n", reply_completion)

======= 流式多图调用 =======
✅ 服务器健康状态：{'status': 'healthy', 'model_loaded': True, 'gpu_count': 4, 'cuda_available': True, 'model_name': 'Qwen/Qwen3-VL-30B-A3B-Instruct'}
📸 已加载图片 1/2：test.png
📸 已加载图片 2/2：test1.jpg
📡 正在调用服务器：http://10.120.17.131:8000/v1/chat/completions/stream
🔍 响应状态码：200
是对两张图片的详细描述和对比。

 图片内容描述

**第一张图片：**
这张照片捕捉了一只活泼可爱的白色博美犬ranian）在户外奔跑的瞬间。这只小狗全身覆盖着蓬松、洁白的毛发，显得非常柔软。它正朝着镜头的方向跑来，一只前爪高高抬起，充满了动感。它的嘴巴微微张开，露出粉红色的舌头，眼睛又大又黑，闪烁着兴奋和好奇的光芒。小狗脖子上系着一条蓝色的项圈或围巾。背景是模糊的森林或公园环境，阳光透过树叶洒下，在绿色的草地和落叶上形成斑驳的光影，营造出一种温暖而自然的氛围。整个画面构图以小狗为中心，焦点清晰，突出了其生动的姿态和快乐的表情。

**第二张图片：**
这张照片展示了一只灰色的英国短毛猫（British Shorthair），它身处一片繁茂的花丛中。猫咪戴着一顶小巧的草编遮阳帽，帽子斜戴在头上，增添了几分俏皮感。它还穿着一件亮黄色的胸背带式牵引绳，表明它可能正在被主人带着散步。猫咪有着标志性的圆脸、宽大的绿色眼睛和长长的胡须，眼神平静地望向一侧，表情略显严肃和沉思。它身后的背景是盛开的粉色马缨丹（Verbena）花朵和翠绿的草地，色彩鲜艳，生机盎然。与第一张照片相比，这张照片的场景更像一个精心布置的花园，光线明亮柔和。

---

### 异同点对比

| 对比维度 | 第一张图片 (博美犬) | 第二张图片 (英短猫) |
| :--- | :--- | :--- |
| **主体动物** | 一只白色的博美犬 | 一只灰色的英国短毛猫 |
| **姿态与动态** | 动态十足，正在奔跑，充满活力和喜悦。 | 静止状态，站立不动，神态平静甚至有些呆萌。 |
| **面部表情** | 兴奋、开心，嘴巴微张，眼

In [ ]:
from client import PhysicalAICenterClient

if __name__ == "__main__":
    # ====================== 测试配置 =======================
    # 初始化客户端（可自定义配置参数）
    client = PhysicalAICenterClient(
        ip="10.120.17.131",
        port=8001,
        model_name="Qwen/Qwen3-VL-2B-Instruct",
        timeout=60,
        max_tokens=2048
    )
    
    # 多图输入示例
    test_image_paths = ["test.png", "test1.jpg"]  # 替换为你的多张图片路径
    test_prompt = "请详细描述这两张图片的内容，并对比它们的异同点"
    
    # 模式1：流式输出（默认）
    print("======= 流式多图调用 =======")
    reply_stream = client.call_qwen_vl_multi_image(test_prompt, test_image_paths, stream=True)
    
    # 模式2：非流式输出
    # print("======= 非流式多图调用 =======")
    # reply_completion = client.call_qwen_vl_multi_image(test_prompt, test_image_paths, stream=False)
    # print("🤖 非流式回复：\n", reply_completion)

======= 流式多图调用 =======
✅ 服务器健康状态：{'status': 'healthy', 'model_loaded': True, 'gpu_count': 4, 'cuda_available': True, 'model_name': 'Qwen/Qwen3-VL-2B-Instruct'}
📸 已加载图片 1/2：test.png
📸 已加载图片 2/2：test1.jpg
📡 正在调用服务器：http://10.120.17.131:8001/v1/chat/completions/stream
🔍 响应状态码：200
，这是一份对您的两张图片的详细描述及对比分析。



### 图片内容描述

**第一张图片：**
- **主体**：一只毛茸茸、蓬松的白色小狗（看起来像一种小型博美犬或类似品种），它正开心地在户外奔跑。
- **姿态与表情**：小狗前爪抬起，仿佛正在跳跃或向前奔跑。它的嘴巴张开，露出粉红色的舌头和牙齿，眼神明亮而充满活力，显得非常兴奋和快乐。
- **环境**：背景是模糊的树林或公园景象，阳光透过树叶洒下斑驳的光影。前景是清晰的绿色草地，草叶上还沾有落叶。
- **细节**：小狗脖子上系着一条蓝色的布质项圈或围巾，为画面增添了一抹亮色。

**第二张图片：**
- **主体**：一只灰白色的英国短毛猫，它戴着一顶编织的草帽，眼睛大而圆，直视镜头。
- **姿态与表情**：猫咪的姿势略显警觉，耳朵微微竖起，但眼神中透露出一丝好奇和惊讶。它身上穿着一个黄色的牵引背带，表明它可能是在被牵着散步。
- **环境**：猫咪身处一片盛开的粉色花朵丛中，这些花可能是美女樱或类似的品种。背景中的绿植和鲜艳的花朵营造出一个生机勃勃的夏日花园场景。
- **细节**：猫咪的胡须很长，整体造型优雅而可爱。

---

### 异同点对比

| 对比维度 | 第一张图片 (小狗) | 第二张图片 (猫咪) |
| :--- | :--- | |
| **核心主题** | 一只活泼可爱的宠物狗在自然环境中玩耍。 | 一只佩戴帽子、显得文静的家养猫在花园里。 |
| **动物种类** | 小型犬（如博美）。 | 猫（具体品种未明确，但为短毛猫）。 |
| **主要特征** | 毛发浓密蓬松，动作生动活泼，表情天真烂漫。 | 身体修长，带有精致的草

## TTS 模型测试

In [3]:
import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel
import os

# 设置环境变量 HF_HOME

os.environ["HF_HOME"] = "/mnt/slurmfs-A100_msp/user_data/dpeng108/data/huggingface_cache"

model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)




Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 11328.30it/s]


In [8]:
# single inference
wavs, sr = model.generate_custom_voice(
    text="其实我真的有发现，我是一个特别善于观察别人情绪的人。",
    language="Chinese", # Pass `Auto` (or omit) for auto language adaptive; if the target language is known, set it explicitly.
    speaker="Vivian",
    instruct="用特别愤怒的语气说", # Omit if not needed.
)
sf.write("output_custom_voice.wav", wavs[0], sr)


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


In [ ]:
# batch inference
wavs, sr = model.generate_custom_voice(
    text=[
        "其实我真的有发现，我是一个特别善于观察别人情绪的人。", 
        "She said she would be here by noon."
    ],
    language=["Chinese", "English"],
    speaker=["Vivian", "Ryan"],
    instruct=["", "Very happy."]
)
sf.write("output_custom_voice_1.wav", wavs[0], sr)
sf.write("output_custom_voice_2.wav", wavs[1], sr)

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


## TTS 服务调用测试

In [ ]:
# pip install fastapi uvicorn pydantic soundfile requests pydub sounddevice numpy pyaudio uvloop
# sudo apt-get install -y portaudio19-dev

### test-01

In [ ]:
import requests
import sounddevice as sd
import numpy as np
import io
from pydub import AudioSegment
from pydub.playback import play

# 服务地址
BASE_URL = "http://10.120.17.131:8000"

def test_full_audio():
    """测试获取完整音频文件"""
    url = f"{BASE_URL}/tts/full"
    data = {
        "text": "其实我真的有发现，我是一个特别善于观察别人情绪的人。",
        "language": "Chinese",
        "speaker": "Vivian",
        "instruct": "用特别愤怒的语气说"
    }
    
    response = requests.post(url, json=data)
    if response.status_code == 200:
        # 保存音频文件
        with open("full_output.wav", "wb") as f:
            f.write(response.content)
        print("完整音频已保存为 full_output.wav")
        
        # 播放音频
        audio = AudioSegment.from_wav(io.BytesIO(response.content))
        play(audio)
    else:
        print(f"请求失败: {response.status_code} - {response.text}")


def test_batch_audio():
    """测试批量生成音频"""
    url = f"{BASE_URL}/tts/batch"
    data = {
        "texts": [
            "其实我真的有发现，我是一个特别善于观察别人情绪的人。",
            "She said she would be here by noon."
        ],
        "languages": ["Chinese", "English"],
        "speakers": ["Vivian", "Ryan"],
        "instructs": ["", "Very happy."]
    }
    
    response = requests.post(url, json=data)
    if response.status_code == 200:
        results = response.json()
        for i, result in enumerate(results["results"]):
            # 解码十六进制音频数据
            audio_bytes = bytes.fromhex(result["audio_data"])
            # 保存文件
            with open(f"batch_output_{i}.wav", "wb") as f:
                f.write(audio_bytes)
            print(f"批量音频 {i} 已保存为 batch_output_{i}.wav")
    else:
        print(f"请求失败: {response.status_code} - {response.text}")

if __name__ == "__main__":
    # 安装依赖：pip install requests pydub sounddevice numpy
    # 需要安装 ffmpeg: apt install ffmpeg 或 brew install ffmpeg
    
    # # 测试完整音频
    test_full_audio()
    
    # 测试批量生成
    # test_batch_audio()

### test-02

In [ ]:
import requests
import pyaudio
import wave
import io
import threading
import queue

# 服务地址
BASE_URL = "http://localhost:8000"
CHUNK = 1024  # 播放缓冲区大小

# 音频播放队列（用于解耦接收和播放）
audio_queue = queue.Queue()
is_playing = False

def audio_player():
    """音频播放线程：实时播放队列中的音频数据"""
    global is_playing
    p = pyaudio.PyAudio()
    
    # 初始化音频流（先获取格式信息）
    stream = None
    
    try:
        while is_playing or not audio_queue.empty():
            try:
                # 从队列获取音频块（超时等待）
                audio_chunk = audio_queue.get(timeout=1.0)
                
                if not audio_chunk:
                    continue
                
                # 解析 WAV 块获取格式信息
                wav_io = io.BytesIO(audio_chunk)
                with wave.open(wav_io, 'rb') as wf:
                    channels = wf.getnchannels()
                    sample_width = wf.getsampwidth()
                    sample_rate = wf.getframerate()
                    
                    # 初始化播放流（首次）
                    if stream is None:
                        stream = p.open(
                            format=p.get_format_from_width(sample_width),
                            channels=channels,
                            rate=sample_rate,
                            output=True,
                            frames_per_buffer=CHUNK
                        )
                    
                    # 读取音频数据并播放
                    data = wf.readframes(CHUNK)
                    while data:
                        stream.write(data)
                        data = wf.readframes(CHUNK)
                
                audio_queue.task_done()
                
            except queue.Empty:
                continue
            except Exception as e:
                print(f"播放错误: {e}")
                continue
                
    finally:
        if stream:
            stream.stop_stream()
            stream.close()
        p.terminate()
        is_playing = False
        print("播放结束")

def test_real_time_stream():
    """测试真正的实时流式播放"""
    global is_playing
    
    # 准备请求数据
    url = f"{BASE_URL}/tts/stream"
    data = {
        "text": "这是一个真正的流式语音合成测试，你应该能听到声音实时播放，而不是等待全部生成完。这句话比较长，可以更好地测试流式效果，每一个字都应该实时地播放出来。这是一个真正的流式语音合成测试，你应该能听到声音实时播放，而不是等待全部生成完。这句话比较长，可以更好地测试流式效果，每一个字都应该实时地播放出来。这是一个真正的流式语音合成测试，你应该能听到声音实时播放，而不是等待全部生成完。这句话比较长，可以更好地测试流式效果，每一个字都应该实时地播放出来。",
        "language": "Chinese",
        "speaker": "Vivian",
        "instruct": "用自然的语气，正常语速说",
        "chunk_size": 1024
    }
    
    # 启动播放线程
    is_playing = True
    player_thread = threading.Thread(target=audio_player)
    player_thread.start()
    
    try:
        print("开始流式请求并播放...")
        # 发送流式请求（关键：stream=True）
        response = requests.post(
            url,
            json=data,
            stream=True,
            # 关键配置：关闭请求缓冲
            headers={"Connection": "keep-alive"},
            timeout=None
        )
        
        if response.status_code != 200:
            print(f"请求失败: {response.status_code} - {response.text}")
            is_playing = False
            return
        
        # 实时接收音频块并放入播放队列
        for chunk in response.iter_content(chunk_size=4096):
            if chunk:
                audio_queue.put(chunk)
        
    except KeyboardInterrupt:
        print("用户中断播放")
    except Exception as e:
        print(f"请求错误: {e}")
    finally:
        # 等待播放完成
        is_playing = False
        player_thread.join()
        print("流式播放测试完成")

def test_full_audio():
    """测试完整音频生成（对比）"""
    url = f"{BASE_URL}/tts/full"
    data = {
        "text": "这是完整生成的音频，需要等待全部生成完才能播放。",
        "language": "Chinese",
        "speaker": "Vivian"
    }
    
    print("开始生成完整音频...")
    response = requests.post(url, json=data)
    if response.status_code == 200:
        # 保存并播放
        with open("full_test.wav", "wb") as f:
            f.write(response.content)
        print("完整音频已保存，开始播放...")
        
        # 播放完整音频
        wf = wave.open("full_test.wav", 'rb')
        p = pyaudio.PyAudio()
        stream = p.open(
            format=p.get_format_from_width(wf.getsampwidth()),
            channels=wf.getnchannels(),
            rate=wf.getframerate(),
            output=True
        )
        
        data = wf.readframes(CHUNK)
        while data:
            stream.write(data)
            data = wf.readframes(CHUNK)
        
        stream.stop_stream()
        stream.close()
        p.terminate()
    else:
        print(f"请求失败: {response.status_code} - {response.text}")

if __name__ == "__main__":
    # 安装依赖
    # pip install requests pyaudio wave
    
    # 测试真正的流式播放（推荐）
    test_real_time_stream()
    
    # 测试完整生成（对比）
    # test_full_audio()

### test-03

In [ ]:
import requests
import pyaudio
import queue
import threading
import time
import wave
import io

# ====================== 配置项 ======================
BASE_URL = "http://localhost:8000"
CHUNK_MS = 20  # 与服务端保持一致
SAMPLE_RATE = 16000  # Qwen3-TTS 默认采样率
SAMPLE_WIDTH = 2  # 16-bit 音频
CHANNELS = 1  # 单声道

# 全局播放队列（双缓冲，避免卡顿）
audio_queue = queue.Queue(maxsize=5)  # 最大缓存5块，防止堆积
is_playing = False

# ====================== 音频播放核心函数 ======================
def audio_player():
    """
    零缓冲音频播放线程：
    - 预配置音频格式，避免动态解析
    - 双缓冲队列，平衡接收和播放速度
    """
    global is_playing
    p = pyaudio.PyAudio()

    # 预初始化播放流（固定格式，减少延迟）
    stream = p.open(
        format=p.get_format_from_width(SAMPLE_WIDTH),
        channels=CHANNELS,
        rate=SAMPLE_RATE,
        output=True,
        frames_per_buffer=int(SAMPLE_RATE * CHUNK_MS / 1000),
        start=False  # 先不启动，等有数据再启动
    )

    try:
        while is_playing or not audio_queue.empty():
            try:
                # 非阻塞获取音频块，超时时间匹配音频块时长
                audio_chunk = audio_queue.get(timeout=CHUNK_MS / 1000 * 2)
                
                # 解析 WAV 块，提取裸 PCM 数据（跳过 WAV 头）
                wav_io = io.BytesIO(audio_chunk)
                with wave.open(wav_io, 'rb') as wf:
                    pcm_data = wf.readframes(wf.getnframes())
                
                # 启动播放流（首次）
                if not stream.is_active():
                    stream.start_stream()
                
                # 实时播放 PCM 数据（零缓冲）
                stream.write(pcm_data)

            except queue.Empty:
                continue
            except Exception as e:
                print(f"播放错误: {e}")
                continue
    finally:
        # 清理资源
        if stream.is_active():
            stream.stop_stream()
        stream.close()
        p.terminate()
        is_playing = False
        print("流式播放结束")

# ====================== 测试函数 ======================
def test_low_latency_stream():
    """测试低延迟流式播放"""
    global is_playing

    # 长文本测试（能明显体现流式效果）
    test_text = """
    这是一个用于测试低延迟流式语音合成的长文本。我们将长文本切分为多个短句，
    逐句生成并实时推送。这样做的好处是，不需要等待整个长文本生成完毕，
    就能听到第一句的声音，极大降低了用户的等待时间。
    而且通过零缓冲传输和实时播放，能够有效避免卡顿问题。
    """

    # 构造请求数据
    data = {
        "text": test_text.strip(),
        "language": "Chinese",
        "speaker": "Vivian",
        "instruct": "",
        "sentence_split": True,  # 开启分句推理
        "chunk_ms": CHUNK_MS
    }

    # 启动播放线程
    is_playing = True
    player_thread = threading.Thread(target=audio_player)
    player_thread.start()

    try:
        print("开始低延迟流式请求...")
        start_time = time.time()
        
        # 发送流式请求（关键配置：禁用缓冲）
        response = requests.post(
            f"{BASE_URL}/tts/stream",
            json=data,
            stream=True,
            headers={
                "Connection": "keep-alive",
                "Accept-Encoding": "identity"  # 禁用压缩，避免缓冲
            },
            timeout=None
        )

        if response.status_code != 200:
            print(f"请求失败: {response.status_code} - {response.text}")
            is_playing = False
            return

        # 实时接收音频块，放入播放队列（无缓冲）
        for chunk in response.iter_content(chunk_size=1024):
            if chunk:
                # 队列满时阻塞，避免数据堆积导致卡顿
                audio_queue.put(chunk, block=True, timeout=1.0)

        # 计算首包延迟
        first_packet_time = time.time() - start_time
        print(f"首包延迟: {first_packet_time:.2f} 秒")

    except KeyboardInterrupt:
        print("用户中断播放")
    except Exception as e:
        print(f"请求错误: {e}")
    finally:
        # 等待播放完成
        is_playing = False
        player_thread.join()
        total_time = time.time() - start_time
        print(f"总耗时: {total_time:.2f} 秒")

if __name__ == "__main__":
    # 安装依赖
    # pip install requests pyaudio
    test_low_latency_stream()